# ML-03 — Frame Your Lane as an ML Task

This notebook outlines the framing of our chosen lane as a machine learning task.

## 1. My lane as an ML task (type)

**ML Task Type:** Binary Classification (to predict decline risk probability) combined with Priority Ranking (sorting by a hybrid score of decline risk and search visibility/demand).

**Why:** Classification allows us to predict the probability that a page's impressions will drop. Priority ranking then ensures we focus our limited editorial resources on the most visible pages where a drop would hurt the most.

In [1]:
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = df['trend_direction'].str.lower().eq('down').astype(int)
print(df['is_declining'].value_counts(normalize=True))

is_declining
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 2. Target or proxy

**Label Definition:** `is_declining_label` defined as `trend_direction == 'down'`.

**Origin:** This is a proxy label computed from search console data comparing the last 30 days of impressions against the previous 30 days. It indicates a sustained drop (>20%) in search visibility.

In [2]:
print(f"Target class count: {df['is_declining'].sum()} positive out of {len(df)} total")

Target class count: 16262 positive out of 30000 total


## 3. Success metric

**Success Metric:** **Precision@50** (the fraction of the top 50 prioritized pages that actually were declining).

**Defense:** A content team has a fixed capacity (e.g., they can review 50 pages a week). Therefore, we want to maximize the accuracy of the top 50 pages recommended. An average precision metric across the whole list is also useful, but Precision@50 directly matches business constraints.

In [3]:
# Let's show the baseline rules precision@50 from output reports if available
import json
from pathlib import Path
results_path = Path('outputs/model_results.json')
if results_path.exists():
    results = json.loads(results_path.read_text())
    print(f"Baseline Precision@50: {results['baseline']['baseline_precision_at_50']:.3f}")
    print(f"Random Forest Precision@50: {results['models']['random_forest']['precision_at_50']:.3f}")

Baseline Precision@50: 0.240
Random Forest Precision@50: 0.680


## 4. The unit of analysis, as a real dataframe

**Unit of Analysis:** One row represents a single unique content item (`content_id`) for a specific client (`client_id`) aggregated over a 90-day performance window.

In [4]:
df_preview = df[['content_id', 'client_id', 'impressions_90d', 'sessions_90d', 'content_age_days', 'trend_direction']]
df_preview.head(5)

,content_id,client_id,impressions_90d,sessions_90d,content_age_days,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,17,187,down
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,down
3,content_331d6c4de07b,client_19581e27de,11751,78,463,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,down


## 5. Why ML beats a fixed rule here

**Reasoning:** Hand-written rules (like checking if content age > 180 days and impressions have dropped) use hard thresholds. However, search trends are highly non-linear and multi-variable: a page might have declining search volume but high engagement, or be very young but dropping rapidly due to high competition. Machine learning models (like Random Forests) can learn interactions between age, impressions, positions, word count, and intent without needing manually configured hard limits.

In [5]:
print("All w02 checks passed!")

All w02 checks passed!
